In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import torch

from drn import Constant, GLM, split_data, preprocess_data

from hyperparameter_tuning_objectives import (
    objective_cann,
    objective_mdn,
    objective_ddr,
    objective_drn,
)

from analysis_utils import rank_models_per_seed, calculate_metrics

In [ ]:
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {accelerator} for training.")

In [ ]:
DATA_DIR = Path("data/interim/real")
features = pd.read_csv(
    DATA_DIR / "features.csv", na_values=["NA"], keep_default_na=False
)
target = pd.read_csv(DATA_DIR / "target.csv")

PLOT_DIR = Path("plots/splitting-test")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Training

In [ ]:
cat_features = [
    "HasKmLimit",
    "Garage",
    "Gender",
    "MariStat",
    "SocioCateg",
    "VehBody",
    "VehEngine",
    "VehEnergy",
    "VehClass",
    "VehPrice",
    "VehUsage",
]

num_features = [feature for feature in features.columns if feature not in cat_features]

In [ ]:
np.random.seed(31052025)

NUM_RANDOM_SPLITS = 100
split_seeds = [int(s) for s in np.random.randint(0, 2**32 - 1, size=NUM_RANDOM_SPLITS)]

distribution = "gamma"

batch_size = 128
patience = 20
lr = 1e-3

hidden_size = 256
dropout_rate = 0.2
num_hidden_layers = 3

num_components = 5

proportion = 0.5

min_obs = 5
kl_alpha = 1e-2
mean_alpha = 1e-2
dv_alpha = 1e-2
kl_direction = "forwards"
criteria = "CRPS"

results_batches = []

for i, split_seed in enumerate(split_seeds):
    print(f"Seed: {split_seed}")
    print(
        "-----------------------------------------------------------------------------------"
    )
    x_train_raw, x_val_raw, x_test_raw, y_train, y_val, y_test = split_data(
        features, target, seed=split_seed
    )

    X_train, X_val, X_test, ct, _ = preprocess_data(
        x_train_raw,
        x_val_raw,
        x_test_raw,
        num_features=num_features,
        cat_features=cat_features,
        num_standard=True,
    )

    constant = Constant(distribution).fit(X_train, y_train)
    glm = GLM(distribution).fit(X_train, y_train)

    print(constant(X_test).mean(), glm(X_test).mean())

    cann_glm = CANN(
        glm,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        batch_size=batch_size,
        distribution=distribution,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    cann_constant = CANN(
        constant,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        batch_size=batch_size,
        distribution=distribution,
        constant=True,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    mdn = MDN(
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        num_components=num_components,
        batch_size=batch_size,
        distribution=distribution,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    ddr = DDR(
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        proportion=0.2,
        batch_size=batch_size,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    drn_glm = DRN(
        glm,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        kl_alpha=kl_alpha,
        mean_alpha=mean_alpha,
        dv_alpha=dv_alpha,
        batch_size=batch_size,
        proportion=proportion,
        min_obs=min_obs,
        distribution=distribution,
        kl_direction=kl_direction,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    drn_constant = DRN(
        constant,
        num_hidden_layers=num_hidden_layers,
        hidden_size=hidden_size,
        dropout_rate=dropout_rate,
        lr=lr,
        kl_alpha=kl_alpha,
        mean_alpha=mean_alpha,
        dv_alpha=dv_alpha,
        batch_size=batch_size,
        proportion=proportion,
        min_obs=min_obs,
        distribution=distribution,
        kl_direction=kl_direction,
    ).fit(X_train, y_train, X_val, y_val, accelerator=accelerator, patience=patience)

    X_test = torch.Tensor(X_test.values)
    Y_test = torch.Tensor(y_test.values).flatten()

    results_batch = calculate_metrics(
        models=[
            drn_constant,
            drn_glm,
            cann_constant,
            cann_glm,
            mdn,
            ddr,
            constant,
            glm,
        ],
        names=[
            "DRN_CONSTANT",
            "DRN_GLM",
            "CANN_CONSTANT",
            "CANN_GLM",
            "MDN",
            "DDR",
            "CONSTANT",
            "GLM",
        ],
        X_test_data=X_test,
        Y_test_data=Y_test,
        y_train=y_train,
        train_size=X_train.shape[0],
        seed_index=i,
    )
    display(results_batch)
    results_batches.append(results_batch)

results = pd.concat(results_batches)
results.to_csv(PLOT_DIR / "rank_distribution_data.csv", index=False)

# Visualisation

In [ ]:
results = pd.read_csv(PLOT_DIR / "rank_distribution_data.csv")

In [ ]:
# Get a subset of the 'results' DataFrame that isn't "CONSTANT" or "GLM" in the 'model' column
results = results[results["model"] != "CONSTANT"]
results = results[results["model"] != "GLM"]

In [ ]:
ranks_nll = rank_models_per_seed(results, "NLL")
ranks_crps = rank_models_per_seed(results, "CRPS")
ranks_rmse = rank_models_per_seed(results, "RMSE")
ranks_ql90 = rank_models_per_seed(results, "QL90")

In [ ]:
metric_ranks = {
    "NLL": ranks_nll,
    "CRPS": ranks_crps,
    "RMSE": ranks_rmse,
    "QL90": ranks_ql90,
}

In [ ]:
# Plotting side-by-side histograms at each x-tick (bar plot style)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

rank_bins = [1, 2, 3]  # 4, 5]
bar_width = 0.15
model_list = list(ranks_nll.columns)
x = np.arange(len(rank_bins))

for ax, (metric, rank_df) in zip(axes, metric_ranks.items()):
    for i, model in enumerate(model_list):
        counts = (
            rank_df[model].value_counts().reindex(rank_bins, fill_value=0).sort_index()
        )
        ax.bar(x + i * bar_width, counts.values, width=bar_width, label=model)

    ax.set_title(f"{metric} Rank Distribution")
    ax.set_xlabel("Rank (1 = Best)")
    ax.set_ylabel("Frequency")
    ax.set_xticks(x + bar_width * 2)
    ax.set_xticklabels(rank_bins)
    ax.legend()
    ax.grid(True, axis="y")

plt.tight_layout()
plt.savefig(PLOT_DIR / "rank_distributions.png")

In [ ]:
# Collect stats for all metrics
summary_stats = []

for metric_name, df in metric_ranks.items():
    for model in df.columns:
        summary_stats.append(
            {
                "Metric": metric_name,
                "Model": model,
                "Mean": df[model].mean(),
                "Std": df[model].std(),
            }
        )

summary_df = pd.DataFrame(summary_stats)

# Unique models and metrics for plotting
models = [
    "CANN_GLM",
    "CANN_CONSTANT",
    "MDN",
    "DDR",
    "DRN_GLM",
    "DRN_CONSTANT",
]  ##summary_df["Model"].unique()
metrics = summary_df["Metric"].unique()
colors = dict(zip(models, sns.color_palette(n_colors=len(models))))

# Manually assign colors to each model
colors = {
    "DRN_CONSTANT": "red",
    "DRN_GLM": "orange",
    "CANN_CONSTANT": "blue",
    "CANN_GLM": "teal",
    "MDN": "black",
    "DDR": "gray",
    # "CONSTANT": "gray",
    # "GLM": "gray",
}

# Regenerate the plot with larger title and legend font sizes
plt.figure(figsize=(10, 5), dpi=200)  # High resolution

offset = 0.1  # horizontal spacing between model points
for i, metric in enumerate(metrics):
    for j, model in enumerate(models):
        row = summary_df[
            (summary_df["Metric"] == metric) & (summary_df["Model"] == model)
        ].iloc[0]
        x = i + (j - len(models) / 2) * offset
        color = colors[model]
        plt.plot(
            [x, x],
            [row["Mean"] - row["Std"], row["Mean"] + row["Std"]],
            color=color,
            linewidth=3,
        )
        plt.plot(x, row["Mean"], "o", color=color, label=model, markersize=12)

# Deduplicate legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(
    by_label.values(),
    by_label.keys(),
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=18,
)

plt.xticks(ticks=range(len(metrics)), labels=metrics, fontsize=15)
plt.yticks(fontsize=15)
plt.title("Model Ranking Across Evaluation Metrics", fontsize=18)
plt.ylabel("Rank (1 = Best)", fontsize=15)
plt.ylim(1, 7.0)
plt.tight_layout()
plt.savefig(PLOT_DIR / "model_ranking_summary.png")